# D1 — Before/After Evaluation Harness

Loads **Llama-3.2-3B-Instruct** in 4-bit (same `BitsAndBytesConfig` as the smoke test /
training notebooks), runs it on a combined set of **25 evaluation prompts**:

- 15 held-out examples from `data/processed/test.jsonl` (never seen during training)
- 8 prompts from `data/baseline_prompts.json` (1 per category, also used in the A2 smoke test)
- 2 general-knowledge "sanity check" prompts (added directly in this notebook) to check for
  catastrophic forgetting

Then it loads the **LoRA adapter** from Story C2 on top of the *same* in-memory base model
and runs the same 25 prompts again. The result is `comparison.json` — a side-by-side
base-vs-fine-tuned report used for scoring in Story D2.

**Before running:**
1. Llama-3.2 is gated — make sure the `HF_TOKEN` Colab secret is set (key icon in the left
   sidebar), same as for the smoke test / training notebooks.
2. GPU runtime: `Runtime > Change runtime type > T4 GPU` (or better).
3. Upload these files into this Colab session's working directory (Files sidebar):
   - `data/processed/test.jsonl`
   - `data/baseline_prompts.json`
   - `models/lora-adapter.zip` (the adapter saved/downloaded at the end of Story C2)
4. If you hit the `numpy.dtype size changed, may indicate binary incompatibility` error
   right after the install cell, use `Runtime > Restart session`, then **skip the
   pip-install cell** and resume from the imports cell — see DEVLOG 2026-06-14 for the
   full explanation (this is the same issue hit during C2 training).

**Decoding:** generation here uses **greedy decoding** (`do_sample=False`), unlike the
smoke test's sampling (`temperature=0.7`). For an eval comparison we want the base and
fine-tuned responses to be reproducible and not differ just due to random sampling.

In [ ]:
# Same version pins as 02_train_qlora.ipynb (Story C2), minus trl/datasets which
# aren't needed for inference-only eval. bitsandbytes is left unpinned (>=0.45.0
# floor) since it needs to match whatever CUDA version Colab's current image ships --
# see DEVLOG 2026-06-14 (bitsandbytes==0.43.3 was incompatible with CUDA 12.8).
!pip install -q transformers==4.44.2 accelerate==0.33.0 "bitsandbytes>=0.45.0" peft==0.12.0 sentencepiece

In [ ]:
import time
import json
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
ADAPTER_DIR = "lora-adapter"
ADAPTER_ZIP = "lora-adapter.zip"

MAX_NEW_TOKENS = 768

## Build the evaluation prompt set (25 prompts)

Upload `test.jsonl` (from `data/processed/`) and `baseline_prompts.json` (from `data/`) into
this Colab session's working directory before running the next cell.

Each prompt is normalized to `{id, category, instruction, expected_output}` —
`expected_output` is the gold `output` field for `test.jsonl` examples, and `None` for the
baseline prompts and sanity prompts (which have no fixed "correct" answer).

In [ ]:
with open("test.jsonl") as f:
    test_examples = [json.loads(line) for line in f]

with open("baseline_prompts.json") as f:
    baseline_prompts = json.load(f)

eval_prompts = []

for ex in test_examples:
    instruction = ex["instruction"]
    if ex.get("input"):
        instruction += "\n\n" + ex["input"]
    eval_prompts.append({
        "id": ex["id"],
        "category": ex["category"],
        "instruction": instruction,
        "expected_output": ex["output"],
    })

for p in baseline_prompts:
    eval_prompts.append({
        "id": p["id"],
        "category": p["category"],
        "instruction": p["instruction"],
        "expected_output": None,
    })

# Story D1 step 6: catastrophic-forgetting sanity check -- general-knowledge prompts
# that aren't in any dataset file, added directly here.
eval_prompts.append({
    "id": "sanity-capital-france",
    "category": "general_knowledge",
    "instruction": "What is the capital of France?",
    "expected_output": None,
})
eval_prompts.append({
    "id": "sanity-haiku-autumn",
    "category": "general_knowledge",
    "instruction": "Write a haiku about autumn.",
    "expected_output": None,
})

print(f"Total eval prompts: {len(eval_prompts)}")
for p in eval_prompts:
    print(f"  {p['id']:35s} {p['category']}")

## Load the base model in 4-bit

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()

load_time = time.time() - t0
print(f"Base model loaded in {load_time:.1f}s")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
def generate(model, instruction: str, max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    messages = [{"role": "user", "content": instruction}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    response_ids = output_ids[0][inputs.shape[-1]:]
    return tokenizer.decode(response_ids, skip_special_tokens=True)


def run_all(model, label):
    """Run every eval prompt through `model`, returning {id: {response, gen_time_seconds}}."""
    responses = {}
    for p in eval_prompts:
        t0 = time.time()
        text = generate(model, p["instruction"])
        gen_time = time.time() - t0
        responses[p["id"]] = {"response": text, "gen_time_seconds": round(gen_time, 1)}
        print(f"[{label}] {p['id']} ({gen_time:.1f}s)")
    return responses

## Run the base model on all 25 prompts

This is the "before" pass — 25 prompts at up to 768 new tokens each, greedy decoding on a
T4, will take a while (expect roughly 10-20+ minutes depending on how long responses run).
Each prompt's time is printed as it completes so you can gauge progress.

In [ ]:
t0 = time.time()
base_responses = run_all(model, "base")
print(f"\nBase model: {len(base_responses)} responses in {(time.time() - t0) / 60:.1f} min")

## Load the LoRA adapter on top of the base model

`PeftModel.from_pretrained` wraps the **same in-memory 4-bit base model** with the LoRA
adapter layers — there's no need to reload the ~6.4GB base weights a second time. If
`lora-adapter/` isn't already present in this session (e.g. fresh runtime), it's unzipped
from the uploaded `lora-adapter.zip`.

In [ ]:
if not os.path.isdir(ADAPTER_DIR):
    !unzip -q {ADAPTER_ZIP}

!ls {ADAPTER_DIR}

model = PeftModel.from_pretrained(model, ADAPTER_DIR)
model.eval()
print("Adapter loaded.")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Run the fine-tuned model on the same 25 prompts

This is the "after" pass — same prompts, same decoding settings, only the adapter differs.

In [ ]:
t0 = time.time()
finetuned_responses = run_all(model, "finetuned")
print(f"\nFine-tuned model: {len(finetuned_responses)} responses in {(time.time() - t0) / 60:.1f} min")

## Save the side-by-side comparison report

Writes `comparison.json` — one entry per prompt with `id`, `category`, `instruction`,
`expected_output` (where available), `base_model_response`, and
`finetuned_model_response`. This is the input to Story D2 scoring.

In [ ]:
comparison = []
for p in eval_prompts:
    comparison.append({
        "id": p["id"],
        "category": p["category"],
        "instruction": p["instruction"],
        "expected_output": p["expected_output"],
        "base_model_response": base_responses[p["id"]]["response"],
        "base_gen_time_seconds": base_responses[p["id"]]["gen_time_seconds"],
        "finetuned_model_response": finetuned_responses[p["id"]]["response"],
        "finetuned_gen_time_seconds": finetuned_responses[p["id"]]["gen_time_seconds"],
    })

with open("comparison.json", "w") as f:
    json.dump(comparison, f, indent=2)

print(f"Saved comparison.json with {len(comparison)} prompts")
print(f"\nModel: {MODEL_ID}")
print(f"Base model load time: {load_time:.1f}s")

In [ ]:
from google.colab import files
files.download("comparison.json")

## Notes

After running, copy `comparison.json` into `eval/results/` in the repo (replacing the
`.gitkeep`), and fill in below for the DEVLOG (Story D1):

- **Base model load time / GPU memory:** _TBD_
- **Total gen time, base pass (25 prompts):** _TBD_
- **Total gen time, fine-tuned pass (25 prompts):** _TBD_
- **GPU memory after adapter load:** _TBD_
- **Any compatibility issues encountered:** _TBD_
- **Any responses that looked empty/truncated/repetitive?** (if so, note which `id`s — may
  indicate `MAX_NEW_TOKENS=768` was hit, or a generation loop)